# Search-11d — Descente sous budget (C#)

**Navigation** : [Search-11c](Search-11c-Empirical-Algorithm-Selection.ipynb) | [Version Python](Search-11d-Descente-Sous-Budget.ipynb) | [Search-12](Search-12-BranchAndBound-Borne-LP-Python.ipynb)

**Objectifs** : distinguer arrêt sur budget et optimum local, mesurer la qualité face à un optimum exact, et relier la descente au théorème Lean. **Prérequis** : boucles, sac à dos 0/1. **Durée** : 20 minutes. Les algorithmes sont pédagogiques et utilisent la BCL.

## 1. Loi de descente

Dans `mimo_lean/Descent.lean`, `cost : σ → Nat` décroît strictement pour chaque flip accepté ; `descent_flips_le_barrier` borne leur nombre, et `descent_target_before_ceiling` suppose en outre un run terminal et `hnostall`. Ici, pour valeurs entières, le coût naturel `floor(LP) - valeur` est non négatif sur les états admissibles. Un budget d'évaluations compte aussi les voisins rejetés et ne certifie pas la terminalité.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
Console.WriteLine($"Runtime {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription}; BCL");

Runtime .NET 10.0.14; BCL


## 2. Sac à dos et descente instrumentée

Le majorant fractionnaire trie les objets par densité ; la programmation dynamique fournit l'optimum entier. On examine tous les voisins 1-flip, sauf si le budget interrompt le balayage.

In [2]:
record Instance(int[] P, int[] V, int C);
record Resultat(int[] X, int Valeur, int Pas, int Evaluations, string Raison, int Strict, int Barriere, double LP);
static class Sac {
 public static Instance Instance(int n, int seed) {
  var r = new Random(seed); var p = Enumerable.Range(0,n).Select(_ => r.Next(10,101)).ToArray();
  var v = p.Select(w => w+r.Next(-5,21)).ToArray(); return new(p,v,(int)(0.5*p.Sum()));
 }
 public static double LP(Instance a) {
  int reste=a.C; double total=0;
  foreach(int i in Enumerable.Range(0,a.P.Length).OrderByDescending(i => (double)a.V[i]/a.P[i])) {
   int pris=Math.Min(reste,a.P[i]); total+=(double)a.V[i]*pris/a.P[i]; reste-=pris; if(reste==0) break;
  } return total;
 }
 public static int Exact(Instance a) {
  var d=new int[a.C+1]; for(int i=0;i<a.P.Length;i++) for(int c=a.C;c>=a.P[i];c--) d[c]=Math.Max(d[c],d[c-a.P[i]]+a.V[i]); return d[a.C];
 }
 public static int Valeur(int[] x, Instance a) => Enumerable.Range(0,x.Length).Sum(i=>x[i]*a.V[i]);
 public static bool Admissible(int[] x, Instance a) => Enumerable.Range(0,x.Length).Sum(i=>x[i]*a.P[i])<=a.C;
 public static Resultat Descente(int[] depart, Instance a, int budget) {
  var x=(int[])depart.Clone(); int eval=0,pas=0,strict=0,barriere=0; double lp=LP(a); string raison="budget";
  while(eval<budget) {
   int meilleur=0, indice=-1, explores=0; int val=Valeur(x,a);
   for(int i=0;i<x.Length && eval<budget;i++) {
    var y=(int[])x.Clone(); y[i]=1-y[i]; eval++; explores++;
    if(Admissible(y,a)) {int delta=Valeur(y,a)-val; if(delta>meilleur) {meilleur=delta; indice=i;}}
   }
   if(indice<0) {raison=explores==x.Length ? "blocage" : "budget"; break;}
   x[indice]=1-x[indice]; int nouveau=Valeur(x,a); if(nouveau<=val) strict++; if(nouveau>lp+1e-9) barriere++; pas++;
  }
  return new(x,Valeur(x,a),pas,eval,raison,strict,barriere,lp);
 }
 public static int[] Depart(Instance a,int seed) {var rng=new Random(seed); var x=Enumerable.Range(0,a.P.Length).Select(_=>rng.Next(2)).ToArray(); return Admissible(x,a)?x:new int[x.Length];}
 public static double Mediane(IEnumerable<double> xs) {var a=xs.OrderBy(x=>x).ToArray(); return (a[(a.Length-1)/2]+a[a.Length/2])/2;}
}
Console.WriteLine("Sac : LP fractionnaire, DP exacte et descente initialises");

Sac : LP fractionnaire, DP exacte et descente initialises


### Contrôles de frontière et loi mesurée

La première fixture a un voisin améliorant non exploré : le résultat doit être « budget », jamais « blocage ». La frontière exacte sans améliorant est bien un blocage.

In [3]:
var petite=new Instance(new[]{2,1},new[]{1,2},1);
var partiel=Sac.Descente(new[]{0,0},petite,1); var zero=Sac.Descente(new[]{0,0},petite,0); var complet=Sac.Descente(new[]{0,1},petite,2);
if(partiel.Raison!="budget" || zero.Raison!="budget" || complet.Raison!="blocage") throw new Exception("frontiere incorrecte");
var runs=new List<Resultat>();
foreach(int n in new[]{16,24,32}) for(int seed=0;seed<20;seed++) {var a=Sac.Instance(n,seed); var r=Sac.Descente(new int[n],a,8*n*n); if(r.Evaluations>8*n*n || r.Pas>Math.Floor(r.LP)) throw new Exception("borne"); runs.Add(r);}
Console.WriteLine($"fixtures partielles/zero/completes : OK ; {runs.Count} runs ; violations strict={runs.Sum(r=>r.Strict)}, barriere={runs.Sum(r=>r.Barriere)} ; depassements=0");

fixtures partielles/zero/completes : OK ; 60 runs ; violations strict=0, barriere=0 ; depassements=0


Les compteurs portent sur les mouvements observés. La borne de flips résulte de la décroissance entière du coût, pas du nombre de voisins évalués. Un run interrompu reste potentiellement améliorable.

### Exercice 1 — redémarrages

Répartir un budget global entre `k` descentes et renvoyer la meilleure valeur et la somme des évaluations. Comparer sur les mêmes 60 instances, sans supposer que les redémarrages gagnent toujours. La garde teste deux petits cas et le budget ; `null` indique le travail à compléter.

In [4]:
static (int valeur,int evaluations)? Redemarrages(Instance a,int budget,int k) {
 // TODO : tirer k points de depart, executer les descentes et conserver la meilleure.
 return null;
}
var ex1a=Redemarrages(petite,20,2); var ex1b=Redemarrages(new Instance(new[]{1},new[]{7},1),10,1);
if(ex1a is null || ex1b is null) Console.WriteLine("Exercice 1 a completer");
else if(ex1a.Value.valeur!=2 || ex1b.Value.valeur!=7 || ex1a.Value.evaluations>20 || ex1b.Value.evaluations>10 || ex1a.Value.evaluations<=0 || ex1b.Value.evaluations<=0) Console.WriteLine("Exercice 1 : controle echoue");
else Console.WriteLine("Exercice 1 : fixtures valides");

Exercice 1 a completer


## 3. Arrêts et cible

Sur 60 instances à budget 2304, puis 30 à budget 48, on compare les blocages à l'optimum exact. Les départs sont tirés par un seul générateur par graine ; si inadmissible, on repart de zéro.

In [5]:
var stats=new Dictionary<string,int>{{"budget",0},{"cible",0},{"hors cible",0}}; var gaps=new List<double>();
for(int seed=0;seed<60;seed++) {var a=Sac.Instance(24,seed); int opt=Sac.Exact(a); var r=Sac.Descente(Sac.Depart(a,seed),a,2304); string k=r.Raison=="budget"?"budget":r.Valeur==opt?"cible":"hors cible"; stats[k]++; if(k!="cible") gaps.Add(1.0-(double)r.Valeur/opt);}
var serre=new Dictionary<string,int>{{"budget",0},{"blocage",0}};
for(int seed=0;seed<30;seed++) {var a=Sac.Instance(24,seed); serre[Sac.Descente(Sac.Depart(a,seed),a,48).Raison]++;}
Console.WriteLine($"60 runs : budget={stats["budget"]}, cible={stats["cible"]}, hors cible={stats["hors cible"]}; gap median={Sac.Mediane(gaps):P1}, max={gaps.Max():P1}");
Console.WriteLine($"30 runs, budget 48 : budget={serre["budget"]}, blocage={serre["blocage"]}");

60 runs : budget=0, cible=0, hors cible=60; gap median=9.7 %, max=17.7 %


30 runs, budget 48 : budget=30, blocage=0


Un blocage hors cible fournit un contre-exemple à `hnostall` pour ce voisinage. Les interruptions sur budget ne sont pas des certificats d'optimalité locale ; les valeurs numériques dépendent des graines et du générateur C#.

Dans cette exécution C#, les 60 blocages sont hors cible (gap médian 9,7 %) ; les 30 runs à budget 48 s’arrêtent sur budget.

### Exercice 2 — comparer deux barrières

Retourner le plafond de flips à partir de la valeur initiale et du majorant entier. Comparer `floor(LP)` à la somme des valeurs, borne triviale pertinente. Les fixtures distinguent une constante d'une formule.

In [6]:
static int? Plafond(int valeurInitiale,int majorant) {
 // TODO : majorer les flips par le cout naturel initial.
 return null;
}
var e2a=Plafond(3,10); var e2b=Plafond(7,9);
if(e2a is null || e2b is null) Console.WriteLine("Exercice 2 a completer");
else Console.WriteLine(e2a==7 && e2b==2 ? "Exercice 2 : fixtures valides" : "Exercice 2 : controle echoue");

Exercice 2 a completer


## 4. Qualité sous budget

Pour les 30 graines 1000 à 1029, calculer le gap à l'optimum DP, sa médiane, ses quartiles empiriques et le pire cas. Les budgets sont 20, 50, 100, 200, 400 et 800 évaluations.

In [7]:
var medians=new List<double>();
foreach(int budget in new[]{20,50,100,200,400,800}) {
 var gapsBudget=new List<double>();
 for(int seed=0;seed<30;seed++) {var a=Sac.Instance(24,1000+seed); int opt=Sac.Exact(a); var r=Sac.Descente(Sac.Depart(a,2000+seed),a,budget); gapsBudget.Add(1.0-(double)r.Valeur/opt);}
 var q=gapsBudget.OrderBy(x=>x).ToArray(); double med=Sac.Mediane(q); medians.Add(med);
 Console.WriteLine($"budget {budget,4} | median {med:P1} | IQR [{q[7]:P1}, {q[22]:P1}] | pire {q.Max():P1}");
}
for(int i=1;i<medians.Count;i++) Console.WriteLine($"gain median {i}: {medians[i-1]-medians[i]:P1}");

budget   20 | median 84.5 % | IQR [13.4 %, 86.9 %] | pire 88.5 %


budget   50 | median 60.1 % | IQR [10.4 %, 67.3 %] | pire 71.2 %


budget  100 | median 34.3 % | IQR [9.9 %, 42.0 %] | pire 47.3 %


budget  200 | median 9.6 % | IQR [8.1 %, 11.1 %] | pire 16.9 %


budget  400 | median 9.5 % | IQR [7.4 %, 11.1 %] | pire 16.9 %


budget  800 | median 9.5 % | IQR [7.4 %, 11.1 %] | pire 16.9 %


gain median 1: 24.4 %


gain median 2: 25.8 %


gain median 3: 24.7 %


gain median 4: 0.1 %


gain median 5: 0.0 %


Ces mesures concernent cet échantillon ; elles ne garantissent ni rendement décroissant ni domination pour toute instance. Les générateurs Python et C# diffèrent. Un plateau peut provenir de blocages locaux, à vérifier par les statuts d'arrêt.

Le gap médian mesuré passe de 84,5 % à 20 évaluations à 9,5 % à 400 et 800 évaluations.

### Exercice 3 — voisinage un ou deux flips

Implémenter la descente sous le même budget avec l'union des flips simples et des paires. Un voisinage de deux flips *seuls* ne contient pas les flips simples. La première fixture exige un simple flip et la seconde un échange de deux bits.

In [8]:
#nullable enable
static Resultat? DescenteUnOuDeux(int[] x,Instance a,int budget) {
 // TODO : evaluer les voisins de Hamming 1 ou 2 dans le budget.
 return null;
}
var e3a=DescenteUnOuDeux(new[]{0},new Instance(new[]{1},new[]{5},1),5);
var e3b=DescenteUnOuDeux(new[]{1,0},new Instance(new[]{1,1},new[]{1,4},1),10);
if(e3a is null || e3b is null) Console.WriteLine("Exercice 3 a completer");
else Console.WriteLine(e3a.Valeur==5 && e3b.Valeur==4 && e3a.Evaluations<=5 && e3b.Evaluations<=10 && e3a.Evaluations>0 && e3b.Evaluations>0 ? "Exercice 3 : fixtures valides" : "Exercice 3 : controle echoue");

Exercice 3 a completer


## 5. Expérience candidate et limites

Comparer un budget partagé entre deux potentiels au meilleur regard isolé sur les mêmes instances et budgets. Le présent modèle se limite au sac à dos, à une descente déterministe et à des flips simples ; il ne reproduit pas l'étude MEALPy de Search-11.

## À retenir

Un balayage complet sans améliorant certifie un optimum local, pas global. Les évaluations et les flips acceptés sont deux compteurs différents ; la borne LP est un majorant, et l'arrêt sur budget n'est pas un run terminal.

## Vérifiez votre compréhension

<details><summary>Pourquoi un budget épuisé ne prouve-t-il pas le blocage ?</summary>Des voisins peuvent rester inexplorés, ou le dernier mouvement peut créer de nouveaux voisins améliorants.</details>

<details><summary>Quelle borne naturelle relie le sac à dos à Lean ?</summary>Pour des valeurs entières admissibles, floor(LP) moins la valeur de l'état ; elle borne les flips acceptés.</details>

## Pour aller plus loin

Lire `mimo_lean/Descent.lean` pour les hypothèses précises, et [Search-11](Search-11-Metaheuristics.ipynb) pour les métaheuristiques. Reprendre les trois exercices : redémarrages, comparaison des barrières et voisinage élargi.